In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [5]:
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"]  = 10
 
df = pd.read_csv("corporate_ai_adoption_dataset.csv")
 
num_cols = ["ai_adoption_level","ai_investment_usd","automation_rate",
            "cost_savings","revenue_impact","productivity_gain",
            "employee_ai_training_hours","ai_maturity_score","deployment_count"]

In [6]:

print("1. DATASET OVERVIEW")
print(f"  Rows        : {df.shape[0]:,}")
print(f"  Columns     : {df.shape[1]}")
print(f"  Missing     : {df.isnull().sum().sum()}")
print(f"  Duplicates  : {df.duplicated().sum()}")
print(f"  Industries  : {df['industry'].nunique()} — {df['industry'].unique().tolist()}")
print(f"  Countries   : {df['country'].nunique()}")
print(f"  Year range  : {df['year'].min()} – {df['year'].max()}")
print()
print(df.describe().round(2).to_string())
 

1. DATASET OVERVIEW
  Rows        : 200,000
  Columns     : 13
  Missing     : 0
  Duplicates  : 0
  Industries  : 10 — ['Financial Services', 'Agriculture', 'Energy', 'Retail', 'Technology', 'Manufacturing', 'Telecom', 'Education', 'Logistics', 'Healthcare']
  Countries   : 15
  Year range  : 2015 – 2035

            year  ai_adoption_level  ai_investment_usd  automation_rate  cost_savings  revenue_impact  productivity_gain  employee_ai_training_hours  ai_maturity_score  deployment_count
count  200000.00          200000.00          200000.00        200000.00     200000.00    2.000000e+05          200000.00                   200000.00          200000.00         200000.00
mean     2024.99               0.53         4870558.44             0.44    2128917.87    2.591989e+06               0.40                       76.76               6.27             26.34
std         6.05               0.26         3679703.77             0.22    2415688.90    3.972751e+06               0.20              

In [7]:
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
fig.suptitle("Distribution of All Numeric Features", fontsize=14, fontweight="bold")
for ax, col in zip(axes.flatten(), num_cols):
    sns.histplot(df[col], ax=ax, kde=True, color="steelblue", bins=40)
    ax.set_title(col.replace("_", " ").title())
    ax.set_xlabel("")
plt.tight_layout()
plt.savefig("eda_01_distributions.png")
plt.close()
print("\nSaved: eda_01_distributions.png")
 


Saved: eda_01_distributions.png


In [8]:
fig, ax = plt.subplots(figsize=(11, 8))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax, linewidths=0.5)
ax.set_title("Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("eda_02_correlation.png")
plt.close()
print("Saved: eda_02_correlation.png")
 

Saved: eda_02_correlation.png


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("AI Investment & Revenue Impact by Industry", fontsize=13, fontweight="bold")
 
industry_inv = df.groupby("industry")["ai_investment_usd"].mean().sort_values()
industry_inv.plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Avg AI Investment by Industry")
axes[0].set_xlabel("USD")
 
industry_rev = df.groupby("industry")["revenue_impact"].mean().sort_values()
industry_rev.plot(kind="barh", ax=axes[1], color="coral")
axes[1].set_title("Avg Revenue Impact by Industry")
axes[1].set_xlabel("USD")
 
plt.tight_layout()
plt.savefig("eda_03_industry_analysis.png")
plt.close()
print("Saved: eda_03_industry_analysis.png")
 

Saved: eda_03_industry_analysis.png


In [10]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Trends Over Years", fontsize=13, fontweight="bold")
 
yearly = df.groupby("year")[["ai_adoption_level","ai_investment_usd","revenue_impact"]].mean()
 
yearly["ai_adoption_level"].plot(ax=axes[0], marker="o", color="steelblue")
axes[0].set_title("Avg AI Adoption Level")
axes[0].set_xlabel("Year")
 
yearly["ai_investment_usd"].plot(ax=axes[1], marker="o", color="green")
axes[1].set_title("Avg AI Investment (USD)")
axes[1].set_xlabel("Year")
 
yearly["revenue_impact"].plot(ax=axes[2], marker="o", color="coral")
axes[2].set_title("Avg Revenue Impact (USD)")
axes[2].set_xlabel("Year")
 
plt.tight_layout()
plt.savefig("eda_04_yearly_trends.png")
plt.close()
print("Saved: eda_04_yearly_trends.png")
 

Saved: eda_04_yearly_trends.png


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Top 10 Countries Analysis", fontsize=13, fontweight="bold")
 
top_countries = df["country"].value_counts().head(10)
top_countries.plot(kind="barh", ax=axes[0], color="mediumpurple")
axes[0].set_title("Number of Companies by Country")
axes[0].set_xlabel("Count")
 
country_roi = df.copy()
country_roi["roi"] = ((country_roi["revenue_impact"] + country_roi["cost_savings"]
                       - country_roi["ai_investment_usd"]) / country_roi["ai_investment_usd"]) * 100
top10 = top_countries.index.tolist()
country_roi[country_roi["country"].isin(top10)].groupby("country")["roi"].mean().sort_values().plot(
    kind="barh", ax=axes[1], color="teal"
)
axes[1].set_title("Avg ROI by Country (Top 10)")
axes[1].set_xlabel("ROI %")
 
plt.tight_layout()
plt.savefig("eda_05_country_analysis.png")
plt.close()
print("Saved: eda_05_country_analysis.png")
 

Saved: eda_05_country_analysis.png


In [12]:
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
fig.suptitle("Boxplots — Outlier Detection", fontsize=14, fontweight="bold")
for ax, col in zip(axes.flatten(), num_cols):
    sns.boxplot(y=df[col], ax=ax, color="lightblue")
    ax.set_title(col.replace("_", " ").title())
plt.tight_layout()
plt.savefig("eda_06_boxplots.png")
plt.close()
print("Saved: eda_06_boxplots.png")

Saved: eda_06_boxplots.png


In [13]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Key Relationships", fontsize=13, fontweight="bold")
 
sample = df.sample(3000, random_state=42)
 
axes[0].scatter(sample["ai_investment_usd"], sample["revenue_impact"],
                alpha=0.3, color="steelblue", s=10)
axes[0].set_xlabel("AI Investment (USD)")
axes[0].set_ylabel("Revenue Impact (USD)")
axes[0].set_title("Investment vs Revenue")
 
axes[1].scatter(sample["ai_maturity_score"], sample["ai_adoption_level"],
                alpha=0.3, color="coral", s=10)
axes[1].set_xlabel("AI Maturity Score")
axes[1].set_ylabel("AI Adoption Level")
axes[1].set_title("Maturity vs Adoption")
 
axes[2].scatter(sample["automation_rate"], sample["productivity_gain"],
                alpha=0.3, color="green", s=10)
axes[2].set_xlabel("Automation Rate")
axes[2].set_ylabel("Productivity Gain")
axes[2].set_title("Automation vs Productivity")
 
plt.tight_layout()
plt.savefig("eda_07_scatter_plots.png")
plt.close()
print("Saved: eda_07_scatter_plots.png")

Saved: eda_07_scatter_plots.png


In [14]:
fig, ax = plt.subplots(figsize=(12, 6))
pivot = df.groupby("industry")[["ai_adoption_level","ai_maturity_score",
                                 "automation_rate","productivity_gain",
                                 "deployment_count"]].mean().round(2)
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlGnBu", ax=ax, linewidths=0.5)
ax.set_title("Industry-wise Average Metrics", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("eda_08_industry_heatmap.png")
plt.close()
print("Saved: eda_08_industry_heatmap.png")
 

Saved: eda_08_industry_heatmap.png


In [15]:
df["roi"] = ((df["revenue_impact"] + df["cost_savings"] - df["ai_investment_usd"])
             / df["ai_investment_usd"]) * 100
 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("ROI Analysis", fontsize=13, fontweight="bold")
 
sns.histplot(df["roi"], bins=60, kde=True, ax=axes[0], color="mediumpurple")
axes[0].axvline(df["roi"].mean(), color="red", linestyle="--", label=f"Mean: {df['roi'].mean():.1f}%")
axes[0].axvline(0, color="black", linestyle="-", label="Break-even (0%)")
axes[0].legend()
axes[0].set_title("ROI Distribution")
axes[0].set_xlabel("ROI %")
 
df.groupby("industry")["roi"].mean().sort_values().plot(
    kind="barh", ax=axes[1], color="mediumpurple"
)
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_title("Avg ROI by Industry")
axes[1].set_xlabel("ROI %")
 
plt.tight_layout()
plt.savefig("eda_09_roi_analysis.png")
plt.close()
print("Saved: eda_09_roi_analysis.png")
 

Saved: eda_09_roi_analysis.png
